# Transport Maps and Pushforward

A **pushforward** is one of the most fundamental operations in probability: given a distribution $p_X$ and a deterministic function $f$, the pushforward $f_\# p_X$ is the distribution of $Y = f(X)$. ProbPipe makes this a first-class operation via `pushforward(f, dist)`.

This notebook covers:

1. **The `pushforward` op** — pushing distributions through arbitrary functions
2. **Dispatch strategies** — closed-form, change-of-variables, and sampling fallback
3. **The `TransportMap` and `Bijector` abstractions** — structured maps that enable richer dispatch
4. **The pushforward registry** — how rules are matched and how to add your own
5. **`return_joint`** — maintaining joint representations alongside exact marginals
6. **Connection to broadcasting** — how `pushforward` relates to `WorkflowFunction` broadcasting

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import tensorflow_probability.substrates.jax.bijectors as tfb

from probpipe import (
    Normal, LogNormal, MultivariateNormal,
    EmpiricalDistribution, BroadcastDistribution,
    BijectorTransformedDistribution,
    TransportMap, Bijector, TFPBijector,
    PushforwardRule, PushforwardMethod, PushforwardInfo,
    pushforward_registry, pushforward,
    sample, log_prob, mean, variance,
    WorkflowFunction,
)

key = jax.random.PRNGKey(0)

## 1. The `pushforward` Op

The simplest use: push a distribution through any Python callable.

In [ ]:
# Push Normal(0, 1) through f(x) = x^2
result = pushforward(lambda x: x ** 2, Normal(0, 1), key=key)

print(f"Result type: {type(result).__name__}")
print(f"Mean(X^2) where X ~ N(0,1): {float(mean(result)):.3f}  (analytical: 1.0)")
print(f"Number of samples: {result.n}")

When `pushforward` receives a plain callable (not a `TransportMap`), it uses a **sampling fallback**: draw samples from the input distribution, apply the function via `jax.vmap`, and wrap the results as an `EmpiricalDistribution`.

The default sample count is 128 (matching `WorkflowFunction.DEFAULT_N_BROADCAST_SAMPLES`). You can override it:

In [ ]:
# More samples for better approximation
result_5k = pushforward(lambda x: x ** 2, Normal(0, 1), key=key, num_samples=5000)
print(f"5000 samples: mean = {float(mean(result_5k)):.4f}")
print(f"128 samples:  mean = {float(mean(result)):.4f}")

## 2. Dispatch Strategies

ProbPipe's pushforward dispatch tries strategies in priority order. For a given `(map_type, dist_type)` pair, the registry finds the highest-priority matching rule.

| Priority | Strategy | When it fires | Result type |
|----------|----------|---------------|-------------|
| 10+ | **Closed-form** | Specific (map, dist) pair has a known formula | Exact parametric distribution |
| 0 | **Change-of-variables** | Map is a `Bijector` | `BijectorTransformedDistribution` |
| -100 | **Sampling fallback** | Always available | `EmpiricalDistribution` |

### 2.1 Closed-Form: Exp + Normal → LogNormal

In [ ]:
# The built-in ExpNormalRule recognizes this combination
result = pushforward(TFPBijector(tfb.Exp()), Normal(loc=1.0, scale=0.5))

print(f"Result type: {type(result).__name__}")
print(f"Is LogNormal: {isinstance(result, LogNormal)}")
print(f"Parameters: loc={float(result.loc):.1f}, scale={float(result.scale):.1f}")

# Verify: log_prob matches the analytical LogNormal density
x = jnp.array([1.0, 2.0, 3.0])
ref = LogNormal(loc=1.0, scale=0.5)
print(f"\nlog_prob match: {bool(jnp.allclose(log_prob(result, x), log_prob(ref, x)))}")

### 2.2 Change-of-Variables: Any Bijector

When the map is a `Bijector` (invertible with a tractable Jacobian), the change-of-variables formula gives an exact density without sampling.

In [ ]:
# Sigmoid bijector: no closed-form rule, falls through to change-of-variables
result = pushforward(TFPBijector(tfb.Sigmoid()), Normal(loc=0.0, scale=1.0))

print(f"Result type: {type(result).__name__}")
print(f"Support: {result.support}")
print(f"Has exact log_prob: True (change-of-variables formula)")

# Sample and verify support
samples = sample(result, key=key, sample_shape=(1000,))
print(f"All in (0, 1): {bool(jnp.all(samples > 0) & jnp.all(samples < 1))}")

### 2.3 Forcing a Strategy

You can force a specific strategy with the `strategy` parameter. This is useful for benchmarking or when you want a specific representation.

In [ ]:
bij = TFPBijector(tfb.Exp())
dist = Normal(loc=0.0, scale=1.0)

# Force change-of-variables (bypass the closed-form Exp+Normal rule)
result_cov = pushforward(bij, dist, strategy="change_of_variables")
print(f"strategy='change_of_variables': {type(result_cov).__name__}")

# Force sampling fallback
result_samp = pushforward(bij, dist, strategy="sampling", key=key, num_samples=500)
print(f"strategy='sampling':             {type(result_samp).__name__} (n={result_samp.n})")

# Auto (default): picks highest-priority match (closed-form)
result_auto = pushforward(bij, dist)
print(f"strategy=None (auto):            {type(result_auto).__name__}")

### 2.4 Querying Without Executing

The registry's `check()` method tells you *which* strategy would fire without actually computing the pushforward.

In [ ]:
from probpipe import pushforward_registry

# What would happen for Exp + Normal?
info = pushforward_registry.check(TFPBijector(tfb.Exp()), Normal(0, 1))
print(f"Exp + Normal:     feasible={info.feasible}, method={info.method.value}")
print(f"                  description: {info.description}")

# What about Sigmoid + Normal?
info2 = pushforward_registry.check(TFPBijector(tfb.Sigmoid()), Normal(0, 1))
print(f"Sigmoid + Normal: feasible={info2.feasible}, method={info2.method.value}")

# Is there a closed-form rule for a generic map + Normal?
from probpipe.maps.registry import _CallableTransportMap
f = _CallableTransportMap(lambda x: x**2)
info3 = pushforward_registry.check(f, Normal(0, 1), strategy="closed_form")
print(f"lambda + Normal (closed_form): feasible={info3.feasible}")

## 3. Transport Maps and Bijectors

While `pushforward` works with plain callables, wrapping your function in a `TransportMap` or `Bijector` enables richer dispatch.

### 3.1 TransportMap: Any Deterministic Function

A `TransportMap[T, S]` is an abstract base class for deterministic maps $f: T \to S$. Subclass it and implement `forward()`.

In [ ]:
class QuadraticMap(TransportMap):
    """f(x) = a*x^2 + b*x + c."""
    def __init__(self, a: float = 1.0, b: float = 0.0, c: float = 0.0):
        self.a, self.b, self.c = a, b, c

    def forward(self, x):
        return self.a * x**2 + self.b * x + self.c

f = QuadraticMap(a=1.0, b=0.0, c=0.0)

# TransportMap supports __call__ as sugar for forward
print(f"f(3.0) = {float(f(3.0))}")

# pushforward via the map
result = f.pushforward(Normal(0, 1), key=key, num_samples=2000)
print(f"E[X^2] where X ~ N(0,1): {float(mean(result)):.3f}")

# Also works with the top-level op
result2 = pushforward(f, Normal(0, 1), key=key, num_samples=2000)
print(f"Same via pushforward(): {float(mean(result2)):.3f}")

### 3.2 Bijector: Invertible Maps with Jacobians

A `Bijector` extends `TransportMap` with `inverse()` and `forward_log_det_jacobian()`. This enables the change-of-variables formula for exact density computation.

In [ ]:
from probpipe.core.distribution import positive

class ExpBijector(Bijector):
    """Pure-probpipe exp bijector (no TFP dependency)."""

    def forward(self, x):
        return jnp.exp(x)

    def inverse(self, y):
        return jnp.log(y)

    def forward_log_det_jacobian(self, x, event_ndims=0):
        return x  # d/dx exp(x) = exp(x), so log|J| = x

    @property
    def output_constraint(self):
        return positive

bij = ExpBijector()
result = bij.pushforward(Normal(0, 1))
print(f"Custom bijector pushforward: {type(result).__name__}")
print(f"Support: {result.support}")

# Verify density matches LogNormal
x = jnp.array([0.5, 1.0, 2.0])
lp_custom = log_prob(result, x)
lp_ref = log_prob(LogNormal(0, 1), x)
print(f"log_prob matches LogNormal: {bool(jnp.allclose(lp_custom, lp_ref, atol=1e-5))}")

### 3.3 TFPBijector: Wrapping TensorFlow Probability Bijectors

`TFPBijector` wraps any `tfb.Bijector` as a probpipe `Bijector`, mirroring the `TFPDistribution` pattern.

In [ ]:
# Wrap TFP bijectors
exp_bij = TFPBijector(tfb.Exp())
sig_bij = TFPBijector(tfb.Sigmoid())
chain_bij = TFPBijector(tfb.Chain([tfb.Exp(), tfb.Shift(1.0), tfb.Scale(2.0)]))

for bij in [exp_bij, sig_bij, chain_bij]:
    constraint = bij.output_constraint
    print(f"{repr(bij):30s}  output_constraint={constraint}")

## 4. The Pushforward Registry

The registry is the dispatch engine behind `pushforward()`. It matches `(map_type, dist_type)` pairs to rules, trying them in priority order.

### 4.1 Built-in Rules

Three rules ship with probpipe:

| Rule | Priority | Map types | Dist types | Method |
|------|----------|-----------|------------|--------|
| `ExpNormalRule` | 10 | `TFPBijector` (Exp only) | `Normal` | `CLOSED_FORM` |
| `_BijectorChangeOfVariablesRule` | 0 | Any `Bijector` | Any `ArrayDistribution` | `CHANGE_OF_VARIABLES` |
| `_SamplingFallbackRule` | -100 | Any `TransportMap` | Any `Distribution` | `SAMPLE` |

### 4.2 Adding Rules: Three Approaches

There are three ways to add your own rules, depending on the complexity.

#### Approach 1: The `@rule` Decorator (Simplest)

Use the `@pushforward_registry.rule` decorator when the rule is **unconditionally feasible** for a given `(map_type, dist_type)` pair — i.e., whenever the types match, the formula always applies.

In [ ]:
class AffineMap(TransportMap):
    """f(x) = scale * x + shift."""
    def __init__(self, scale: float, shift: float):
        self.scale_val = scale
        self.shift_val = shift

    def forward(self, x):
        return self.scale_val * x + self.shift_val

# Register: AffineMap + Normal -> Normal (always valid)
@pushforward_registry.rule(
    AffineMap, Normal,
    method=PushforwardMethod.CLOSED_FORM,
    priority=15,
    description="Affine transform of Normal is Normal",
)
def _affine_normal(m, d, **kwargs):
    return Normal(
        loc=m.scale_val * d.loc + m.shift_val,
        scale=abs(m.scale_val) * d.scale,
    )

# Test it
result = pushforward(AffineMap(2.0, 3.0), Normal(1.0, 0.5))
print(f"2*N(1, 0.5) + 3 = {type(result).__name__}(loc={float(result.loc):.1f}, scale={float(result.scale):.1f})")
print(f"Expected: Normal(loc=5.0, scale=1.0)")

#### Approach 2: `PushforwardRule` Subclass (Conditional Feasibility)

Subclass `PushforwardRule` when feasibility depends on **runtime values**, not just types. For example, the built-in `ExpNormalRule` checks whether the `TFPBijector` actually wraps a `tfb.Exp()` (vs. a `tfb.Sigmoid()`).

In [ ]:
from probpipe import Gamma

class ScaleGammaRule(PushforwardRule):
    """Scaling a Gamma by a positive constant: c * Gamma(a, b) = Gamma(a, b/c).

    Only feasible when scale > 0.
    """
    def map_types(self):
        return (AffineMap,)

    def dist_types(self):
        return (Gamma,)

    def check(self, transport_map, dist):
        # Only feasible for pure scaling (no shift) with positive scale
        if transport_map.shift_val == 0.0 and transport_map.scale_val > 0:
            return PushforwardInfo(
                feasible=True,
                method=PushforwardMethod.CLOSED_FORM,
                description=f"Scale Gamma by {transport_map.scale_val}",
            )
        return PushforwardInfo(feasible=False)

    def apply(self, transport_map, dist, **kwargs):
        return Gamma(
            concentration=dist.concentration,
            rate=dist.rate / transport_map.scale_val,
        )

    @property
    def priority(self):
        return 20  # Higher than AffineMap+Normal (15)

pushforward_registry.register(ScaleGammaRule())

# Test: 3 * Gamma(2, 1) = Gamma(2, 1/3)
result = pushforward(AffineMap(3.0, 0.0), Gamma(concentration=2.0, rate=1.0))
print(f"3 * Gamma(2, 1) = {type(result).__name__}")
print(f"  concentration={float(result.concentration):.1f}, rate={float(result.rate):.4f}")
print(f"  mean = {float(mean(result)):.1f} (expected: 6.0)")

# With shift != 0, falls through to sampling
result2 = pushforward(AffineMap(3.0, 1.0), Gamma(concentration=2.0, rate=1.0), key=key)
print(f"\n3 * Gamma(2,1) + 1 (shift≠0): {type(result2).__name__} (no closed form)")

#### Approach 3: Direct Registration with `register()`

You can also instantiate a `PushforwardRule` subclass and register it directly. This is useful when the rule is defined in a separate module or package.

In [ ]:
# The ScaleGammaRule above already demonstrated this pattern:
# pushforward_registry.register(ScaleGammaRule())
#
# The @rule decorator is sugar for:
#   1. Create a _FunctionalRule wrapping the function
#   2. Call pushforward_registry.register(rule)
#
# Use register() directly when:
#   - The rule has conditional feasibility (needs check())
#   - The rule is defined in another module
#   - You want to store a reference to the rule object

# Summary of when to use each approach:
print("""
When to use each approach:

  @rule decorator      →  Simple type-pair rules, always feasible
  PushforwardRule      →  Conditional feasibility (runtime checks)
  register()           →  Same as PushforwardRule, for external modules
""")

## 5. `return_joint`: Joint Representations

By default, `pushforward` returns only the output distribution. With `return_joint=True`, it returns a `BroadcastDistribution` that stores **both** representations:

- **Paired input–output samples** (the joint)
- **The exact output marginal** (when a closed-form or change-of-variables rule was used)

This dual representation is powerful: you get the exact analytical output distribution *and* the ability to trace which inputs produced which outputs.

In [ ]:
# Closed-form + return_joint
joint = pushforward(
    TFPBijector(tfb.Exp()), Normal(loc=0.0, scale=1.0),
    return_joint=True, key=key,
)

print(f"Type: {type(joint).__name__}")
print(f"Component names: {joint.component_names}")
print(f"Number of paired samples: {joint.n}")

# The exact output marginal is stored
exact_output = joint._exact_output_marginal
print(f"\nExact output marginal: {type(exact_output).__name__}")
print(f"  loc={float(exact_output.loc):.1f}, scale={float(exact_output.scale):.1f}")

# Access components
input_emp = joint["input"]
output_emp = joint["_output"]
print(f"\nInput samples shape: {input_emp.samples.shape}")
print(f"Output samples shape: {output_emp.samples.shape}")

In [ ]:
# Visualize the joint: input-output correspondence
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Left: input distribution
axes[0].hist(input_emp.samples, bins=40, density=True, alpha=0.7)
axes[0].set_title("Input: Normal(0, 1)")
axes[0].set_xlabel("x")

# Middle: scatter of (input, output) pairs
axes[1].scatter(input_emp.samples, output_emp.samples, alpha=0.3, s=5)
axes[1].set_xlabel("input x")
axes[1].set_ylabel("output exp(x)")
axes[1].set_title("Joint: (x, exp(x))")

# Right: output distribution vs exact
axes[2].hist(output_emp.samples, bins=40, density=True, alpha=0.7, label="empirical")
x_grid = jnp.linspace(0.01, 8.0, 200)
axes[2].plot(x_grid, jnp.exp(jax.vmap(lambda v: log_prob(exact_output, v))(x_grid)),
             'r-', linewidth=2, label="exact LogNormal")
axes[2].set_title("Output: exp(Normal) = LogNormal")
axes[2].set_xlabel("y")
axes[2].set_xlim(0, 8)
axes[2].legend()

plt.tight_layout()
plt.show()

### Sampling Fallback with `return_joint`

When using the sampling fallback (no closed-form or CoV rule), `return_joint=True` still returns a `BroadcastDistribution`, but without an exact output marginal. The joint is purely empirical.

In [ ]:
# Non-invertible function: no exact output distribution possible
joint_samp = pushforward(
    lambda x: jnp.sin(x) * jnp.exp(-x**2 / 4),
    Normal(0, 2),
    return_joint=True, key=key, num_samples=500,
)

print(f"Has exact marginal: {hasattr(joint_samp, '_exact_output_marginal')}")
print(f"Paired samples: {joint_samp.n}")

# Visualize
input_s = joint_samp["input"].samples
output_s = joint_samp["_output"].samples

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.scatter(input_s, output_s, alpha=0.3, s=5)
ax1.set_xlabel("x")
ax1.set_ylabel("sin(x) * exp(-x²/4)")
ax1.set_title("Joint pushforward (empirical)")

ax2.hist(output_s, bins=40, density=True, alpha=0.7, color="orange")
ax2.set_title("Output marginal")
ax2.set_xlabel("y")
plt.tight_layout()
plt.show()

## 6. Connection to Broadcasting

ProbPipe has two complementary systems for pushing distributions through functions:

| Feature | `pushforward()` | `WorkflowFunction` broadcasting |
|---------|-----------------|----------------------------------|
| **Input** | `(function, distribution)` | Function called with distribution args |
| **Exact dispatch** | Registry of closed-form and CoV rules | Always sampling-based |
| **Joint return** | `return_joint=True` | `include_inputs=True` |
| **Multi-input** | Single input distribution | Multiple broadcast arguments |
| **Orchestration** | No Prefect support | Prefect task/flow wrapping |

`pushforward` is specialized for the single-input case and can exploit mathematical structure (bijector inverses, closed-form formulas). `WorkflowFunction` broadcasting is general-purpose: it handles multiple distribution inputs, non-JAX functions (via loop backend), and Prefect orchestration.

### Using pushforward with WorkflowFunction

Since `pushforward` is itself a probpipe op (a `WorkflowFunction`), it participates in the standard op infrastructure.

In [ ]:
# pushforward as a WorkflowFunction: use it like any other op
bij = TFPBijector(tfb.Exp())

# Direct call
result = pushforward(bij, Normal(0, 1))
print(f"Direct: {type(result).__name__}")

# Broadcasting comparison: WorkflowFunction wraps the function for general use
def my_transform(x: float) -> float:
    return jnp.exp(x)

wf = WorkflowFunction(func=my_transform, vectorize="jax", n_broadcast_samples=128, seed=42)
wf_result = wf(x=Normal(0, 1))
print(f"WF broadcast: {type(wf_result).__name__}")
print(f"  mean via pushforward (exact LogNormal): {float(mean(result)):.4f}")
print(f"  mean via WF broadcast (empirical):      {float(mean(wf_result)):.4f}")
print()
print("Key difference: pushforward recognized Exp+Normal and returned")
print("an exact LogNormal; WF broadcasting always samples.")

## 7. Provenance Tracking

All pushforward results automatically record provenance, regardless of strategy.

In [ ]:
from probpipe import provenance_ancestors

# Closed-form
cf = pushforward(TFPBijector(tfb.Exp()), Normal(0, 1, name="prior"))
print(f"Closed-form provenance:")
print(f"  operation: {cf.source.operation}")
print(f"  method:    {cf.source.metadata['method']}")
print(f"  rule:      {cf.source.metadata.get('rule', 'N/A')}")
print(f"  parent:    {cf.source.parents[0].name}")

print()

# Sampling fallback
sf = pushforward(lambda x: x**2, Normal(0, 1, name="base"), key=key)
print(f"Sampling provenance:")
print(f"  operation:   {sf.source.operation}")
print(f"  method:      {sf.source.metadata['method']}")
print(f"  num_samples: {sf.source.metadata['num_samples']}")

## Summary

**Key takeaways:**

1. **`pushforward(f, dist)`** is the primary API — works with plain callables, `TransportMap` subclasses, and `Bijector` subclasses.

2. **Three dispatch strategies** in priority order: closed-form rules > change-of-variables (bijectors) > sampling fallback.

3. **`TransportMap`** provides structure for deterministic maps; **`Bijector`** adds invertibility for exact density.

4. **Registry customization** via three approaches:
   - `@pushforward_registry.rule(MapType, DistType)` — simple, always-feasible rules
   - `PushforwardRule` subclass with `check()`/`apply()` — conditional feasibility
   - `pushforward_registry.register(rule)` — for rules defined in external modules

5. **`return_joint=True`** returns a `BroadcastDistribution` that maintains both the exact output marginal (when available) and paired input–output samples.

6. **`pushforward` vs. broadcasting**: `pushforward` exploits mathematical structure for single-input maps; `WorkflowFunction` broadcasting handles general multi-input workflows.